# Day 027 Project: DailyBriefingScheduler

## What You're Building

A `DailyBriefingScheduler` class that:
1. Holds a list of job configs (added via `add_job`)
2. Checks which jobs are due using `is_due` + a persisted run log
3. Generates an AI briefing for due jobs
4. Saves the briefing to a file and updates the run log

This extends Day 1's AI Briefing Generator with automated scheduling and idempotency.

## Project Requirements

1. Implement `DailyBriefingScheduler` with:
   - `add_job(name, fn, interval_minutes)` — add a job config
   - `due_jobs(now=None) -> list[str]` — names of due jobs
   - `save_briefing(content, output_dir) -> str` — write to dated file
   - `run(topics, output_dir, model, now) -> dict` — full pipeline
2. Run `scheduler.run(TOPICS, '/tmp')` and store as `result`
3. Verify with `_run_project_checks()`

In [ ]:
import json, os, tempfile
from datetime import datetime
from pathlib import Path
import ollama

## Provided: All Helper Functions

In [ ]:
def build_job(
    name: str,
    fn,
    interval_minutes: int,
    enabled: bool = True,
) -> dict:
    if interval_minutes <= 0:
        raise ValueError(f"interval_minutes must be > 0, got {interval_minutes}")
    return {
        "name": name,
        "fn_name": fn.__name__,
        "interval_minutes": interval_minutes,
        "enabled": enabled,
    }


def is_due(
    last_run_iso: str | None,
    interval_minutes: int,
    now: datetime | None = None,
) -> bool:
    if last_run_iso is None:
        return True
    if now is None:
        now = datetime.now()
    last_run = datetime.fromisoformat(last_run_iso)
    elapsed_seconds = (now - last_run).total_seconds()
    return elapsed_seconds >= interval_minutes * 60


def save_run_log(path: str, records: list[dict]) -> None:
    Path(path).write_text(json.dumps(records, indent=2), encoding="utf-8")


def load_run_log(path: str) -> list[dict]:
    p = Path(path)
    if not p.exists():
        return []
    return json.loads(p.read_text(encoding="utf-8"))


def record_run(
    log: list[dict],
    name: str,
    result: str,
    status: str = "ok",
) -> list[dict]:
    new_record = {
        "name": name,
        "ran_at": datetime.now().isoformat(),
        "result": result,
        "status": status,
    }
    return log + [new_record]


def ai_daily_briefing(topics: list[str], model: str = "llama3.2") -> str:
    topics_str = "\n".join(f"- {t}" for t in topics)
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a professional daily briefing assistant. "
                    "Write a concise, structured briefing covering the given topics. "
                    "Use clear section headers (## Topic). Keep it under 300 words."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Generate a daily briefing covering these topics:\n{topics_str}\n\n"
                    "Today's briefing:"
                ),
            },
        ],
    )
    return response["message"]["content"]

## Your Implementation

Implement `DailyBriefingScheduler` by wiring the helper functions.

In [ ]:
class DailyBriefingScheduler:
    def __init__(self, log_path: str = '/tmp/day027_run_log.json'):
        self.log_path = log_path
        self.jobs: list[dict] = []

    def add_job(self, name: str, fn, interval_minutes: int) -> None:
        # TODO: self.jobs.append(build_job(name, fn, interval_minutes))
        pass

    def _last_run(self, name: str) -> str | None:
        # TODO: load_run_log; filter by name; return last ran_at or None
        pass

    def due_jobs(self, now: datetime | None = None) -> list[str]:
        # TODO: [j['name'] for j in self.jobs if j['enabled'] and
        #        is_due(self._last_run(j['name']), j['interval_minutes'], now)]
        pass

    def save_briefing(self, content: str, output_dir: str) -> str:
        # TODO: today = datetime.now().strftime('%Y-%m-%d')
        # TODO: path = str(Path(output_dir) / f'briefing_{today}.txt')
        # TODO: Path(path).write_text(content, encoding='utf-8')
        # TODO: return path
        pass

    def run(
        self, topics: list[str], output_dir: str,
        model: str = 'llama3.2', now: datetime | None = None,
    ) -> dict:
        # TODO: due = self.due_jobs(now)
        # TODO: content = ai_daily_briefing(topics, model=model)
        # TODO: path = self.save_briefing(content, output_dir)
        # TODO: log = load_run_log(self.log_path)
        # TODO: for name in due: log = record_run(log, name, f'saved to {path}')
        # TODO: save_run_log(self.log_path, log)
        # TODO: return {'content': content, 'path': path, 'ran_jobs': due}
        pass

## Topics and Run

In [ ]:
TOPICS = [
    'AI Engineering learning progress',
    'Project status update',
    'Today\'s priorities',
]


In [ ]:
# def my_briefing_task(): pass  # placeholder callable
# scheduler = DailyBriefingScheduler()
# scheduler.add_job('daily_briefing', my_briefing_task, interval_minutes=1440)
# result = scheduler.run(TOPICS, '/tmp')
# print(f"Briefing saved to: {result['path']}")
# print(f"Ran jobs: {result['ran_jobs']}")

## Plug into APScheduler (Optional Extension)

Once `DailyBriefingScheduler.run` works, you can wrap it in APScheduler to run automatically:

```python
from apscheduler.schedulers.blocking import BlockingScheduler

aps = BlockingScheduler()

@aps.scheduled_job('cron', hour=7, minute=30)
def morning_briefing():
    result = scheduler.run(TOPICS, '/tmp')
    print(f"Briefing saved: {result['path']}")

# aps.start()  # blocks until stopped; run in a terminal, not a notebook
```

Or interval-based:

```python
@aps.scheduled_job('interval', hours=1)
def hourly_check():
    ...
```


## Checks

In [ ]:
def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: DailyBriefingScheduler has all required methods
    try:
        assert 'DailyBriefingScheduler' in globals()
        for m in ('add_job', 'due_jobs', 'save_briefing', 'run'):
            assert hasattr(DailyBriefingScheduler, m), \
                f'DailyBriefingScheduler missing: {m}'
        passed += 1; print('\u2705 Check 1: all methods present')
    except Exception as e:
        print(f'\u274c Check 1: {e}')

    # Check 2: scheduler is an instance
    try:
        assert 'scheduler' in globals()
        assert isinstance(scheduler, DailyBriefingScheduler)
        passed += 1; print('\u2705 Check 2: scheduler is DailyBriefingScheduler')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: result dict has required keys
    try:
        assert 'result' in globals()
        for k in ('content', 'path', 'ran_jobs'):
            assert k in result, f"result missing '{k}': {list(result)}"
        passed += 1; print('\u2705 Check 3: result has content/path/ran_jobs')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: briefing file exists
    try:
        assert 'result' in globals()
        path = result.get('path', '')
        assert os.path.exists(path), f'briefing file not found: {path}'
        size = os.path.getsize(path)
        assert size > 20, f'briefing file too small ({size} bytes)'
        passed += 1; print(f'\u2705 Check 4: briefing file exists ({size} bytes)')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: briefing content is non-empty string
    try:
        assert 'result' in globals()
        content = result.get('content', '')
        assert isinstance(content, str) and len(content) > 20, \
            f'content should be non-empty str: {content!r}'
        passed += 1; print(f'\u2705 Check 5: briefing is {len(content)} chars')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Project complete!')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()

## Bonus Challenges

- Add a `disable_job(name)` method that sets `enabled=False` for a named job
- Add a `status()` method that returns a dict of `{job_name: last_ran, is_due}` for all jobs
- Persist jobs to a JSON config file (separate from the run log) so the scheduler reloads its job list on restart
- Add an `error_handler(fn)` callback: if `ai_daily_briefing` raises, call the handler and record `status='error'` in the run log
- Wire it to APScheduler's cron trigger: fire at 07:30 every weekday
- Add an `only_weekdays` flag to `build_job` and check it in `due_jobs`